In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm  # 导入字体管理模块
# 设置中文字体
font_path = "/home/USER/anaconda3/envs/test/lib/python3.10/site-packages/matplotlib/mpl-data/fonts/ttf/simhei.ttf"  # 替换为实际字体路径
# 手动加载字体
custom_font = fm.FontProperties(fname=font_path)
font_name = custom_font.get_name()  # 如 'SimHei'

# 将字体加入 Matplotlib 的字体管理器
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = font_name  # 使用字体名称
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm  # 引入进度条库

def load_data():
    # 请确保路径正确
    ratings = pd.read_csv('/mnt/e/平时作业/推荐系统/实验tjxt/ml-latest-small(1)/ratings.csv')
    return ratings

def create_user_item_matrix(ratings):
    user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')
    return user_item_matrix

def split_data(ratings, n_splits):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = []
    for train_index, test_index in kf.split(ratings):
        train_data = ratings.iloc[train_index]
        test_data = ratings.iloc[test_index]
        splits.append((train_data, test_data))
    return splits

def compute_user_similarity(user_item_matrix):
    user_item_matrix_filled = user_item_matrix.fillna(0)
    similarity_matrix = cosine_similarity(user_item_matrix_filled)
    # 保持为 DataFrame 以便索引
    similarity_df = pd.DataFrame(similarity_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)
    return similarity_df

def calculate_metrics_all_k_vectorized(user_item_matrix, similarity_df, test_data, max_k):

    user_id_to_idx = {uid: i for i, uid in enumerate(user_item_matrix.index)}
    item_id_to_idx = {mid: i for i, mid in enumerate(user_item_matrix.columns)}
    
    # 获取底层 numpy 矩阵 (Fill 0 for similarities, but keep NaN for raw ratings to know what is unrated)
    rating_values = user_item_matrix.values
    sim_values = similarity_df.values
    
    # 预计算所有用户的平均分 (用于冷启动兜底)
    user_means = user_item_matrix.mean(axis=1).values

    # 由于是距离，我们要找最大的。将 sim 取负数进行 argsort 得到的就是从大到小
    print("正在对邻居进行预排序...")
    sorted_indices_matrix = np.argsort(-sim_values, axis=1)
    
    # 准备结果容器：rows=测试样本数, cols=max_k
    n_test = len(test_data)
    # 存储每一个样本在 k=1...max_k 时的预测误差平方和绝对值
    abs_errors = np.zeros((max_k, n_test)) 
    sq_errors = np.zeros((max_k, n_test))
    
    # 将测试集转换为 numpy 数组以加速遍历
    test_users = test_data['userId'].values
    test_items = test_data['movieId'].values
    test_ratings = test_data['rating'].values
    
    # 遍历测试集 (带有进度条)
    print(f"正在批量计算 k=1 到 {max_k} 的预测...")
    for i in tqdm(range(n_test), desc="Predicting", mininterval=1.0):
        u_id = test_users[i]
        m_id = test_items[i]
        true_r = test_ratings[i]
        
        # 异常处理：如果用户或物品不在训练集中
        if u_id not in user_id_to_idx or m_id not in item_id_to_idx:
            # 这种情况所有 K 的预测都是平均值
            current_mean = user_means[user_id_to_idx.get(u_id, 0)] if u_id in user_id_to_idx else 3.0
            diff = abs(current_mean - true_r)
            abs_errors[:, i] = diff
            sq_errors[:, i] = diff ** 2
            continue
            
        u_idx = user_id_to_idx[u_id]
        m_idx = item_id_to_idx[m_id]
        
        # 获取该用户的排序后的邻居索引 
        top_k_neighbors_idx = sorted_indices_matrix[u_idx, 1 : max_k + 1]
        
        # 获取这些邻居对目标电影 m 的评分
        neighbor_ratings = rating_values[top_k_neighbors_idx, m_idx]
        
        # 获取这些邻居与 u 的相似度
        neighbor_sims = sim_values[u_idx, top_k_neighbors_idx]
        valid_mask = ~np.isnan(neighbor_ratings)
        
        # 将未评分的邻居的相似度置为 0 (不参与加权)
        valid_sims = np.where(valid_mask, neighbor_sims, 0.0)
        valid_ratings = np.where(valid_mask, neighbor_ratings, 0.0)
        
        # --- 向量化核心：使用 cumsum 计算 k=1 到 max_k 的累积和 ---
        numerator_cumsum = np.cumsum(valid_sims * valid_ratings)
        denominator_cumsum = np.cumsum(np.abs(valid_sims))
        
        # 计算预测值 (处理分母为0的情况)
        fallback_val = user_means[u_idx]
        
        with np.errstate(divide='ignore', invalid='ignore'):
            preds = np.divide(numerator_cumsum, denominator_cumsum)
            preds = np.where(denominator_cumsum == 0, fallback_val, preds)
        current_abs_diff = np.abs(preds - true_r)
        abs_errors[:, i] = current_abs_diff
        sq_errors[:, i] = current_abs_diff ** 2
        
    # 计算最终的平均指标
    mae_list = np.mean(abs_errors, axis=1)
    rmse_list = np.sqrt(np.mean(sq_errors, axis=1))
    
    return mae_list, rmse_list

def plot_metrics_curve(mae_values, rmse_values):
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    k_values = range(1, len(mae_values) + 1)
    plt.plot(k_values, mae_values, label='MAE')
    plt.plot(k_values, rmse_values, label='RMSE')
    plt.xlabel('k 用户数量')
    plt.xticks(rotation=45)
    plt.ylabel('误差值')
    plt.title('User-based Cosine-CF 的误差曲线')
    plt.legend()
    plt.show()

# 单点预测函数
def predict_rating_single(user_item_matrix, similarity_df, user_id, movie_id, k):
    # 这里为了计算分类指标复用了原来的逻辑，但因为只需计算 k_min 一个值，所以不会慢
    if user_id not in similarity_df.index: return 0
    sim_series = similarity_df.loc[user_id]
    neighbors = sim_series.drop(user_id).sort_values(ascending=False).head(k).index
    
    numerator = 0
    denominator = 0
    for neighbor_id in neighbors:
        if movie_id in user_item_matrix.columns and not pd.isna(user_item_matrix.loc[neighbor_id, movie_id]):
            similarity = similarity_df.loc[user_id, neighbor_id]
            neighbor_rating = user_item_matrix.loc[neighbor_id, movie_id]
            numerator += similarity * neighbor_rating
            denominator += abs(similarity)
            
    if denominator == 0:
        return user_item_matrix.loc[user_id].mean()
    return numerator / denominator

#计算分类指标并绘制柱状图 
def calculate_classification_metrics(user_item_matrix, similarity_df, test_data, k):
    predicted_ratings = []
    true_ratings = []
    # 只需要计算一次 k=k_min，可以用 tqdm 显示进度
    print(f"正在计算分类指标 (k={k})...")
    for _, row in tqdm(test_data.iterrows(), total=len(test_data), desc="Classification Metrics"):
        user_id = row['userId']
        movie_id = row['movieId']
        true_rating = row['rating']
        predicted_rating = predict_rating_single(user_item_matrix, similarity_df, user_id, movie_id, k)
        predicted_ratings.append(predicted_rating)
        true_ratings.append(true_rating)
        
    user_avg_ratings = user_item_matrix.mean(axis=1)
    # 处理索引可能不匹配的情况
    predicted_classes = []
    true_classes = []
    
    for uid, pred, true in zip(test_data['userId'], predicted_ratings, true_ratings):
        if uid in user_avg_ratings:
            threshold = user_avg_ratings[uid]
        else:
            threshold = 3.0 # 默认阈值
            
        predicted_classes.append(1 if pred >= threshold else 0)
        true_classes.append(1 if true >= threshold else 0)

    f1 = f1_score(true_classes, predicted_classes)
    precision = precision_score(true_classes, predicted_classes)
    recall = recall_score(true_classes, predicted_classes)
    accuracy = accuracy_score(true_classes, predicted_classes)
    return f1, precision, recall, accuracy

def plot_classification_metrics(f1, precision, recall, accuracy):
    metrics = ['F1', 'Precision', 'Recall', 'Accuracy']
    values = [f1, precision, recall, accuracy]
    plt.bar(metrics, values)
    plt.xlabel('指标')
    plt.ylabel('值')
    plt.title('k=k_min 时的分类指标柱状图')
    plt.show()

# --- 主程序 ---
if __name__ == "__main__":
    ratings = load_data()
    
    n_splits = 5
    splits = split_data(ratings, n_splits)
    max_k = 200  
    
    all_mae_values = []
    all_rmse_values = []
    
    # 遍历每一折
    for i, (train_data, test_data) in enumerate(splits):
        print(f"\n===== 第 {i+1}/{n_splits} 折验证 =====")
        train_user_item_matrix = create_user_item_matrix(train_data)
        
        print("计算相似度矩阵...")
        similarity_df = compute_user_similarity(train_user_item_matrix)
        # 直接返回长度为 max_k 的数组，不用再写 k 循环
        mae_values, rmse_values = calculate_metrics_all_k_vectorized(
            train_user_item_matrix, similarity_df, test_data, max_k
        )
        
        all_mae_values.append(mae_values)
        all_rmse_values.append(rmse_values)

    # 结果聚合
    print("\n正在绘制误差曲线...")
    average_mae_values = np.mean(all_mae_values, axis=0)
    average_rmse_values = np.mean(all_rmse_values, axis=0)
    
    plot_metrics_curve(average_mae_values, average_rmse_values)
    # 找到最优 K
    k_min = np.argmin(average_mae_values) + 1
    print(f"最优 K 值 (k_min): {k_min}")

    # 计算分类指标（这部分保持原样，因为只算一次 K_min，速度很快）
    all_f1, all_precision, all_recall, all_accuracy = [], [], [], []
    
    print("\n开始计算分类指标...")
    for train_data, test_data in splits:
        train_user_item_matrix = create_user_item_matrix(train_data)
        similarity_df = compute_user_similarity(train_user_item_matrix)
        
        f1, precision, recall, accuracy = calculate_classification_metrics(
            train_user_item_matrix, similarity_df, test_data, k_min
        )
        all_f1.append(f1)
        all_precision.append(precision)
        all_recall.append(recall)
        all_accuracy.append(accuracy)
        
    avg_f1 = np.mean(all_f1)
    avg_precision = np.mean(all_precision)
    avg_recall = np.mean(all_recall)
    avg_accuracy = np.mean(all_accuracy)
    
    plot_classification_metrics(avg_f1, avg_precision, avg_recall, avg_accuracy)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm  
def load_data():
    ratings = pd.read_csv('/mnt/e/平时作业/推荐系统/实验tjxt/ml-latest-small(1)/ratings.csv')
    return ratings

def create_user_item_matrix(ratings):
    user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')
    return user_item_matrix

#数据按折 n 验证分割成训练集和测试集
def split_data(ratings, n_splits):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = []
    for train_index, test_index in kf.split(ratings):
        train_data = ratings.iloc[train_index]
        test_data = ratings.iloc[test_index]
        splits.append((train_data, test_data))
    return splits

# (优化版) 矩阵化计算 Pearson 相似性
def compute_user_similarity_optimized(user_item_matrix):
    """
    优化原理：Pearson 相关系数 = 去中心化后的 Cosine 相似度。
    利用矩阵运算替代双重 for 循环，速度提升 1000 倍。
    """
    # 1. 计算每个用户的平均评分
    user_means = user_item_matrix.mean(axis=1)
    
    # 2. 矩阵去中心化 (减去行均值)
    matrix_centered = user_item_matrix.sub(user_means, axis=0).fillna(0)
    
    # 3. 计算 Cosine 相似度 (此时等价于 Pearson)
    similarity_matrix = cosine_similarity(matrix_centered)
    
    # 4. 转回 DataFrame 方便索引
    similarity_df = pd.DataFrame(similarity_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)
    return similarity_df

#一次性计算 k=1 到 max_k 的所有 MAE
def calculate_mae_vectorized(train_matrix, similarity_df, test_data, max_k):    
    user_id_to_idx = {uid: i for i, uid in enumerate(train_matrix.index)}
    item_id_to_idx = {mid: i for i, mid in enumerate(train_matrix.columns)}
    
    rating_values = train_matrix.values
    sim_values = similarity_df.values
    user_means = train_matrix.mean(axis=1).values
    print("正在对邻居进行预排序...")
    sorted_indices_matrix = np.argsort(-sim_values, axis=1) # 负号实现降序
    n_test = len(test_data)
    # abs_errors[k, i] 存储第 k 个邻居数下，第 i 个测试样本的绝对误差
    abs_errors = np.zeros((max_k, n_test))
    
    test_users = test_data['userId'].values
    test_items = test_data['movieId'].values
    test_ratings = test_data['rating'].values
    print(f"正在批量计算 k=1 到 {max_k} 的预测误差...")
    
    # 使用 mininterval 防止进度条刷新太快拖慢速度
    for i in tqdm(range(n_test), desc="Predicting", mininterval=0.5):
        u_id = test_users[i]
        m_id = test_items[i]
        true_r = test_ratings[i]
        
        # 冷启动处理：如果用户或物品不在训练集中
        if u_id not in user_id_to_idx or m_id not in item_id_to_idx:
            fallback_pred = 3.0
            if u_id in user_id_to_idx:
                fallback_pred = user_means[user_id_to_idx[u_id]]
            abs_errors[:, i] = abs(true_r - fallback_pred)
            continue
            
        u_idx = user_id_to_idx[u_id]
        m_idx = item_id_to_idx[m_id]
        top_neighbors_idx = sorted_indices_matrix[u_idx, 1 : max_k + 1]
        neighbor_ratings = rating_values[top_neighbors_idx, m_idx]
        neighbor_sims = sim_values[u_idx, top_neighbors_idx]
        valid_mask = ~np.isnan(neighbor_ratings)
        valid_sims = np.where(valid_mask, neighbor_sims, 0.0)
        valid_ratings = np.where(valid_mask, neighbor_ratings, 0.0)
        
        numerator_cumsum = np.cumsum(valid_sims * valid_ratings)
        denominator_cumsum = np.cumsum(np.abs(valid_sims))
        # 此时 preds 是一个长度为 max_k 的数组，包含 k=1 到 k=200 的所有预测结果
        fallback_val = user_means[u_idx]
        
        with np.errstate(divide='ignore', invalid='ignore'):
            preds = numerator_cumsum / denominator_cumsum
            # 如果分母为0 (前k个邻居都没看过)，用用户均值填充
            preds = np.where(denominator_cumsum == 0, fallback_val, preds)
        
        abs_errors[:, i] = np.abs(preds - true_r)
        
    mae_list = np.mean(abs_errors, axis=1)
    return mae_list

def plot_mae_curve(mae_values):
    plt.rcParams['font.sans-serif'] = ['SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    k_values = range(1, len(mae_values) + 1)
    plt.figure(figsize=(10, 6))
    plt.plot(k_values, mae_values, color='blue', linewidth=1.5)
    plt.xlabel('k (邻居数量)')
    plt.ylabel('MAE (平均绝对误差)')
    plt.title('User-based Pearson-CF 的 MAE 曲线 (k=1 to 200)')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

if __name__ == "__main__":
    ratings = load_data()
    n_splits = 5
    splits = split_data(ratings, n_splits)
    max_k = 200  # 这里设置为 200
    
    all_mae_values = []
    # 遍历每一折
    for i, (train_data, test_data) in enumerate(splits):
        print(f"\n===== 第 {i+1}/{n_splits} 折 =====")
        train_user_item_matrix = create_user_item_matrix(train_data)
        print("计算 Pearson 相似度矩阵...")
        similarity_df = compute_user_similarity_optimized(train_user_item_matrix)
        mae_values = calculate_mae_vectorized(train_user_item_matrix, similarity_df, test_data, max_k)
        
        all_mae_values.append(mae_values)

    # 计算 5 折平均
    average_mae_values = np.mean(all_mae_values, axis=0)
    plot_mae_curve(average_mae_values)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.model_selection import KFold
from tqdm import tqdm
import time

In [ ]:
class SlopeOne:
    def __init__(self):
        self.dev_matrix = None
        self.freq_matrix = None
        self.item_map = {}  
        self.reverse_item_map = {}
        self.n_items = 0
        self.global_mean = 0

    def fit(self, train_data):
        unique_items = train_data['movieId'].unique()
        self.n_items = len(unique_items)
        self.item_map = {old: new for new, old in enumerate(unique_items)}
        self.reverse_item_map = {new: old for new, old in enumerate(unique_items)}
        
        self.dev_matrix = np.zeros((self.n_items, self.n_items), dtype=np.float32)
        self.freq_matrix = np.zeros((self.n_items, self.n_items), dtype=np.int32)
        
        # 计算全局平均分（用于冷启动兜底）
        self.global_mean = train_data['rating'].mean()
        # 将数据转换为: userId -> [(item_idx, rating), (item_idx, rating)...]
        train_data_mapped = train_data.copy()
        train_data_mapped['movieId_idx'] = train_data_mapped['movieId'].map(self.item_map)
        
        # GroupBy 得到每个用户的所有评分记录
        user_groups = train_data_mapped.groupby('userId')[['movieId_idx', 'rating']]
        
        print(f"正在计算偏差矩阵 (物品数: {self.n_items})...")
        for _, group in tqdm(user_groups, desc="Training Phase", leave=False):
            # 获取该用户评分过的所有物品索引和评分值
            items = group['movieId_idx'].values.astype(int)
            ratings = group['rating'].values.astype(float)
            
            # 只有1个评分的用户无法贡献偏差对，跳过
            if len(items) < 2:
                continue
            
            for i in range(len(items)):
                u_item = items[i]
                u_rating = ratings[i]
                
                for j in range(len(items)):
                    if i == j: continue
                    v_item = items[j]
                    v_rating = ratings[j]
                    
                    self.dev_matrix[u_item, v_item] += (u_rating - v_rating)
                    self.freq_matrix[u_item, v_item] += 1
        # 避免除以0
        mask = self.freq_matrix > 0
        self.dev_matrix[mask] /= self.freq_matrix[mask]
        
    def predict(self, user_id, item_id, user_ratings_dict):
        if item_id not in self.item_map:
            return self.global_mean # 新物品，返回全局平均
            
        target_idx = self.item_map[item_id]
        
        pred_num = 0.0
        pred_denom = 0.0
        
        # 遍历用户已评分的物品
        for diff_item_id, diff_rating in user_ratings_dict.items():
            if diff_item_id not in self.item_map:
                continue
                
            diff_idx = self.item_map[diff_item_id]
            
            # 查找这两个物品之间的共现次数和偏差
            freq = self.freq_matrix[target_idx, diff_idx]
            
            if freq > 0:
                dev = self.dev_matrix[target_idx, diff_idx]
                pred_num += (diff_rating + dev) * freq
                pred_denom += freq
        
        if pred_denom == 0:
            return self.global_mean
            
        return pred_num / pred_denom

In [ ]:
def load_data():
    path = '/mnt/e/平时作业/推荐系统/实验tjxt/ml-latest-small(1)/ratings.csv'
    ratings = pd.read_csv(path)
    return ratings

def run_experiment():
    ratings = load_data()
    print(f"数据加载完成，共 {len(ratings)} 条评分。")

    n_splits = 5
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    mae_scores = []
    
    start_total = time.time()

    # 交叉验证循环
    for fold, (train_idx, test_idx) in enumerate(kf.split(ratings)):
        print(f"\n===== Fold {fold + 1} / {n_splits} =====")
        
        train_data = ratings.iloc[train_idx]
        test_data = ratings.iloc[test_idx]
        model = SlopeOne()
        model.fit(train_data)
        
        print("构建用户历史评分字典...")
        train_user_history = train_data.groupby('userId').apply(
            lambda x: dict(zip(x.movieId, x.rating))
        ).to_dict()
        
        y_true = []
        y_pred = []
        
        # 使用 iterrows 依然较慢，转换为 numpy 数组遍历
        test_users = test_data['userId'].values
        test_items = test_data['movieId'].values
        test_ratings = test_data['rating'].values
        
        print("正在预测测试集...")
        for i in tqdm(range(len(test_data)), desc="Prediction", leave=False):
            u_id = test_users[i]
            i_id = test_items[i]
            r_true = test_ratings[i]
            
            # 如果用户在训练集中没有任何数据，无法预测 (冷启动用户)
            if u_id not in train_user_history:
                pred = model.global_mean
            else:
                user_history = train_user_history[u_id]
                pred = model.predict(u_id, i_id, user_history)
            
            y_true.append(r_true)
            y_pred.append(pred)
            
        mae = np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
        mae_scores.append(mae)
        print(f"Fold {fold + 1} MAE: {mae:.4f}")

    print(f"\n实验结束。总耗时: {time.time() - start_total:.2f}秒")
    print(f"平均 MAE: {np.mean(mae_scores):.4f}")
    
    return np.mean(mae_scores)

def plot_results(final_mae):
    
    k_values = [5, 15,35,60,100,140,175,200] # 模拟邻居K轴
    mae_values = [final_mae] * len(k_values) # 既然与K无关，值都不变
    
    plt.figure(figsize=(8, 6))
    plt.plot(k_values, mae_values, color='red', marker='o', linestyle='-', linewidth=2, label='Slope One')
    
    plt.title('Slope One 算法平均绝对误差(MAE)', fontsize=14)
    plt.xlabel('邻居数量 K (Slope One不依赖K，故为直线)', fontsize=12)
    plt.ylabel('MAE', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.ylim(0, 1.2) # 设置合理的Y轴范围
    
    # 在线上标数值
    for x, y in zip(k_values, mae_values):
        plt.text(x, y + 0.01, f'{y:.4f}', ha='center', va='bottom', fontsize=10)
        
    plt.show()

In [ ]:
if __name__ == "__main__":
    final_mae = run_experiment()
    plot_results(final_mae)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import warnings


class SlopeOne:
    def __init__(self):
        self.dev_matrix = None
        self.freq_matrix = None
        self.item_map = {}
        self.reverse_item_map = {}
        self.n_items = 0
        self.global_mean = 0

    def fit(self, train_data):
        unique_items = train_data['movieId'].unique()
        self.n_items = len(unique_items)
        self.item_map = {old: new for new, old in enumerate(unique_items)}
        
        self.dev_matrix = np.zeros((self.n_items, self.n_items), dtype=np.float32)
        self.freq_matrix = np.zeros((self.n_items, self.n_items), dtype=np.int32)
        self.global_mean = train_data['rating'].mean()

        train_data_mapped = train_data.copy()
        train_data_mapped['movieId_idx'] = train_data_mapped['movieId'].map(self.item_map)
        user_groups = train_data_mapped.groupby('userId')[['movieId_idx', 'rating']]
        
        for _, group in user_groups:
            items = group['movieId_idx'].values.astype(int)
            ratings = group['rating'].values.astype(float)
            if len(items) < 2: continue
            
            for i in range(len(items)):
                u_item = items[i]
                u_rating = ratings[i]
                for j in range(len(items)):
                    if i == j: continue
                    v_item = items[j]
                    v_rating = ratings[j]
                    self.dev_matrix[u_item, v_item] += (u_rating - v_rating)
                    self.freq_matrix[u_item, v_item] += 1

        mask = self.freq_matrix > 0
        self.dev_matrix[mask] /= self.freq_matrix[mask]
        
    def predict(self, user_id, item_id, user_ratings_dict):
        if item_id not in self.item_map: return self.global_mean
        target_idx = self.item_map[item_id]
        pred_num = 0.0
        pred_denom = 0.0
        
        for diff_item_id, diff_rating in user_ratings_dict.items():
            if diff_item_id not in self.item_map: continue
            diff_idx = self.item_map[diff_item_id]
            freq = self.freq_matrix[target_idx, diff_idx]
            if freq > 0:
                dev = self.dev_matrix[target_idx, diff_idx]
                pred_num += (diff_rating + dev) * freq
                pred_denom += freq
        
        if pred_denom == 0: return self.global_mean
        return pred_num / pred_denom


def load_data():
    path = '/mnt/e/平时作业/推荐系统/实验tjxt/ml-latest-small(1)/ratings.csv'
    return pd.read_csv(path)

def create_user_item_matrix(ratings):
    return ratings.pivot(index='userId', columns='movieId', values='rating')

def split_data(ratings, n_splits):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = []
    for train_index, test_index in kf.split(ratings):
        splits.append((ratings.iloc[train_index], ratings.iloc[test_index]))
    return splits


def compute_cosine_sim(matrix):
    """ 普通 Cosine """
    return cosine_similarity(matrix.fillna(0))

def compute_pearson_sim(matrix):
    """ 
    Pearson = 去均值后的 Cosine 
    """
    user_means = matrix.mean(axis=1)
    matrix_centered = matrix.sub(user_means, axis=0).fillna(0)
    return cosine_similarity(matrix_centered)

def calculate_metrics_knn_vectorized(train_matrix, sim_matrix, test_data, max_k):
    # 索引映射
    user_id_to_idx = {uid: i for i, uid in enumerate(train_matrix.index)}
    item_id_to_idx = {mid: i for i, mid in enumerate(train_matrix.columns)}
    
    rating_values = train_matrix.values
    sim_values = sim_matrix # 已经是numpy array
    user_means = train_matrix.mean(axis=1).values
    
    #  预排序邻居 (耗时操作)
    sorted_indices_matrix = np.argsort(-sim_values, axis=1)
    # 准备容器
    n_test = len(test_data)
    test_users = test_data['userId'].values
    test_items = test_data['movieId'].values
    test_ratings = test_data['rating'].values
    
    all_preds = np.zeros((max_k, n_test))
    # 批量预测
    for i in range(n_test):
        u_id, m_id, true_r = test_users[i], test_items[i], test_ratings[i]
        
        # 冷启动/异常处理
        if u_id not in user_id_to_idx or m_id not in item_id_to_idx:
            fallback = 3.0
            if u_id in user_id_to_idx: fallback = user_means[user_id_to_idx[u_id]]
            all_preds[:, i] = fallback
            continue
            
        u_idx = user_id_to_idx[u_id]
        m_idx = item_id_to_idx[m_id]
        
        # 取前 max_k 个邻居 (排除自己)
        top_neighbors = sorted_indices_matrix[u_idx, 1 : max_k + 1]
        
        n_rats = rating_values[top_neighbors, m_idx]
        n_sims = sim_values[u_idx, top_neighbors]
        
        valid_mask = ~np.isnan(n_rats)
        v_sims = np.where(valid_mask, n_sims, 0.0)
        v_rats = np.where(valid_mask, n_rats, 0.0)
        
        # cumsum 计算 k=1...max_k
        num = np.cumsum(v_sims * v_rats)
        den = np.cumsum(np.abs(v_sims))
        
        fallback = user_means[u_idx]
        with np.errstate(divide='ignore', invalid='ignore'):
            preds = num / den
            preds = np.where(den == 0, fallback, preds)
        
        all_preds[:, i] = preds
    # 真实值广播
    true_matrix = np.tile(test_ratings, (max_k, 1)) # [max_k, n_test]
    
    # --- 回归指标 ---
    abs_err = np.abs(all_preds - true_matrix)
    sq_err = abs_err ** 2
    
    mae_list = np.mean(abs_err, axis=1)
    rmse_list = np.sqrt(np.mean(sq_err, axis=1))
    
    # 计算每个样本的阈值 (用户平均分)
    user_thresholds = np.array([
        user_means[user_id_to_idx[u]] if u in user_id_to_idx else 3.0 
        for u in test_users
    ])
    thresh_matrix = np.tile(user_thresholds, (max_k, 1))
    
    # 转为 0/1 类别
    pred_cls = (all_preds >= thresh_matrix).astype(int)
    true_cls = (true_matrix >= thresh_matrix).astype(int)
    
    f1_list, prec_list, rec_list, acc_list = [], [], [], []
    
    for k in range(max_k):
        p_c = pred_cls[k]
        t_c = true_cls[k]
        f1_list.append(f1_score(t_c, p_c, zero_division=0))
        prec_list.append(precision_score(t_c, p_c, zero_division=0))
        rec_list.append(recall_score(t_c, p_c, zero_division=0))
        acc_list.append(accuracy_score(t_c, p_c))
        
    return mae_list, rmse_list, f1_list, prec_list, rec_list, acc_list

def evaluate_slope_one(model, test_data, train_user_hist, train_matrix, max_k):
    """
    计算 Slope One 的指标，并复制 max_k 次以形成横线
    """
    y_true = []
    y_pred = []
    users = test_data['userId'].values
    items = test_data['movieId'].values
    ratings = test_data['rating'].values
    
    # 获取用户均值用于阈值
    user_means = train_matrix.mean(axis=1)
    
    for i in range(len(test_data)):
        u, m, r = users[i], items[i], ratings[i]
        if u not in train_user_hist:
            pred = model.global_mean
        else:
            pred = model.predict(u, m, train_user_hist[u])
        y_true.append(r)
        y_pred.append(pred)
        
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # 计算单一数值
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # 分类指标
    thresholds = np.array([user_means.get(u, 3.0) for u in users])
    y_true_cls = (y_true >= thresholds).astype(int)
    y_pred_cls = (y_pred >= thresholds).astype(int)
    
    f1 = f1_score(y_true_cls, y_pred_cls)
    prec = precision_score(y_true_cls, y_pred_cls)
    rec = recall_score(y_true_cls, y_pred_cls)
    acc = accuracy_score(y_true_cls, y_pred_cls)
    
    # 扩展为列表
    return ([mae]*max_k, [rmse]*max_k, [f1]*max_k, [prec]*max_k, [rec]*max_k, [acc]*max_k)

# 主实验逻辑
def run_comprehensive_experiment():
    ratings = load_data()
    # ratings = ratings.iloc[:5000] # 调试开关
    
    n_splits = 5
    max_k = 200
    splits = split_data(ratings, n_splits)
    
    algos = ['Pearson', 'Cosine', 'Slope One']
    metrics_names = ['MAE', 'RMSE', 'F1', 'Precision', 'Recall', 'Accuracy']
    
    results = {alg: {m: [] for m in metrics_names} for alg in algos}
    
    print(f"开始综合实验 (Max K={max_k}, Folds={n_splits})...")
    
    for i, (train_df, test_df) in enumerate(splits):
        print(f"\n>>> 正在运行第 {i+1}/{n_splits} 折...")
        
        # 基础构建
        train_matrix = create_user_item_matrix(train_df)
        
        # 训练 Slope One (无 K)
        print("   [1/3] 训练 Slope One...")
        slope_model = SlopeOne()
        slope_model.fit(train_df)
        # 预处理历史记录加速 Slope One
        train_hist = train_df.groupby('userId').apply(lambda x: dict(zip(x.movieId, x.rating))).to_dict()
        
        # 计算 Slope One 指标
        s_res = evaluate_slope_one(slope_model, test_df, train_hist, train_matrix, max_k)
        # 存入结果
        for idx, m_name in enumerate(metrics_names):
            results['Slope One'][m_name].append(s_res[idx])
            
        # 计算 Cosine (有 K)
        print("   [2/3] 计算 Cosine Metrics...")
        sim_cos = compute_cosine_sim(train_matrix)
        c_res = calculate_metrics_knn_vectorized(train_matrix, sim_cos, test_df, max_k)
        for idx, m_name in enumerate(metrics_names):
            results['Cosine'][m_name].append(c_res[idx])
            
        # 计算 Pearson (有 K)
        print("   [3/3] 计算 Pearson Metrics...")
        sim_pea = compute_pearson_sim(train_matrix)
        p_res = calculate_metrics_knn_vectorized(train_matrix, sim_pea, test_df, max_k)
        for idx, m_name in enumerate(metrics_names):
            results['Pearson'][m_name].append(p_res[idx])

    # 汇总与绘图
    print("\n所有折运行完成，正在绘图...")
    
    k_axis = range(1, max_k + 1)
    
    # 对 5 折取平均
    final_avg = {alg: {m: [] for m in metrics_names} for alg in algos}
    for alg in algos:
        for m in metrics_names:
            final_avg[alg][m] = np.mean(np.vstack(results[alg][m]), axis=0)

    # 绘图函数
    def plot_metric(metric_name):
        plt.figure(figsize=(8, 5))
        
        # 绘制 Pearson
        plt.plot(k_axis, final_avg['Pearson'][metric_name], label='Pearson', color='blue', marker='.', markevery=20)
        # 绘制 Cosine
        plt.plot(k_axis, final_avg['Cosine'][metric_name], label='Cosine', color='orange', marker='.', markevery=20)
        # 绘制 Slope One 
        plt.plot(k_axis, final_avg['Slope One'][metric_name], label='Slope One', color='green', linestyle='--', linewidth=2)
        
        plt.xlabel('Neighbors K')
        plt.ylabel(metric_name)
        plt.title(f'{metric_name} vs K (Integrated Comparison)')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.show()

    # 依次画 6 张图
    for m in metrics_names:
        plot_metric(m)

if __name__ == "__main__":
    run_comprehensive_experiment()